# Evaluation Metrics Report

This notebook recomputes non-RAG evaluation metrics from saved result JSON files and ground-truth files in `chunk/test` without rerunning generation.

Inputs:
- `act_full.json`
- `output/reasoning_one_call_eval/final_results.json`
- `output/generation_eval/final_past.json`

The metrics include existing report metrics, stricter law-signature metrics, sentencing metrics, defendant alignment, Toi_Danh exact/partial match rates, field completeness, text overlap, and operational token/fallback metrics. RAG retrieval/context metrics are intentionally excluded.

In [ ]:
from __future__ import annotations

import ast
import json
import math
import re
import unicodedata
from pathlib import Path
from statistics import mean, median
from typing import Any

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from rag.core.sentencing import extract_imprisonment_months

RESULT_FILES = {
    "act": Path("act_full.json"),
    "one_call": Path("output/reasoning_one_call_eval/final_results.json"),
    "past": Path("output/generation_eval/final_past.json"),
}

TEST_DIR = Path("chunk/test")
EXPORT_DIR = Path("output/metrics_report")

MANDATORY_SUPPORT_DIEU = {"38", "50", "51", "52", "53", "54", "55", "56", "57", "58", "65", "47"}
EXTRA_SUPPORT_DIEU = {"46", "48"}
SUPPORT_DIEU = MANDATORY_SUPPORT_DIEU | EXTRA_SUPPORT_DIEU

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 120)
plt.style.use("seaborn-v0_8-whitegrid")

## Helper Functions

In [ ]:
def load_json(path: Path) -> dict[str, Any]:
    with path.open(encoding="utf-8") as fh:
        return json.load(fh)


def normalize_space(value: Any) -> str:
    return re.sub(r"\s+", " ", str(value or "").strip())


def strip_accents(text: str) -> str:
    norm = unicodedata.normalize("NFKD", text)
    return "".join(ch for ch in norm if not unicodedata.combining(ch))


def normalize_text(value: Any) -> str:
    text = strip_accents(normalize_space(value)).lower()
    return re.sub(r"[^a-z0-9]+", " ", text).strip()


def nonempty(value: Any) -> bool:
    if value is None:
        return False
    if isinstance(value, (list, tuple, set, dict)):
        return bool(value)
    text = normalize_space(value)
    return bool(text and text.lower() not in {"none", "null", "[]", "{}"})


def tokenize(value: Any) -> set[str]:
    return {tok for tok in normalize_text(value).split() if tok}


def token_f1(pred: Any, gt: Any) -> float | None:
    pred_tokens = tokenize(pred)
    gt_tokens = tokenize(gt)
    if not pred_tokens and not gt_tokens:
        return None
    if not pred_tokens or not gt_tokens:
        return 0.0
    tp = len(pred_tokens & gt_tokens)
    precision = tp / len(pred_tokens)
    recall = tp / len(gt_tokens)
    return 2 * precision * recall / (precision + recall) if precision + recall else 0.0


TOI_DANH_STOPWORDS = {
    "toi", "ve", "quy", "dinh", "cua", "cac", "bo", "luat", "hinh", "su",
    "theo", "diem", "khoan", "dieu", "va", "hoac", "doi", "voi",
}


def _flatten_toi_danh_values(value: Any) -> list[str]:
    if value is None:
        return []
    if isinstance(value, (list, tuple, set)):
        out: list[str] = []
        for item in value:
            out.extend(_flatten_toi_danh_values(item))
        return out
    text = normalize_space(value)
    if not text:
        return []
    if text.startswith("[") and text.endswith("]"):
        try:
            parsed = ast.literal_eval(text)
            return _flatten_toi_danh_values(parsed)
        except (SyntaxError, ValueError):
            pass
    return [text]


def normalize_toi_danh(value: Any) -> list[str]:
    normalized: list[str] = []
    for raw in _flatten_toi_danh_values(value):
        text = normalize_text(raw)
        text = re.sub(r"^toi\s+", "", text).strip()
        if text:
            normalized.append(text)
    return sorted(set(normalized))


def toi_danh_partial_match(pred: Any, gt: Any) -> bool:
    pred_items = normalize_toi_danh(pred)
    gt_items = normalize_toi_danh(gt)
    if not pred_items or not gt_items:
        return False
    for pred_item in pred_items:
        pred_tokens = tokenize(pred_item) - TOI_DANH_STOPWORDS
        for gt_item in gt_items:
            gt_tokens = tokenize(gt_item) - TOI_DANH_STOPWORDS
            if pred_item == gt_item or pred_item in gt_item or gt_item in pred_item:
                return True
            if pred_tokens and gt_tokens and pred_tokens & gt_tokens:
                return True
    return False


def normalize_signature(value: Any) -> str:
    sig = normalize_space(value)
    sig = re.sub(r"^(dieu|điều|khoan|khoản|diem|điểm)\s+", "", sig, flags=re.IGNORECASE)
    sig = sig.strip(" .")
    sig = re.sub(r"\s+", "", sig)
    return sig


def signature_set(item: dict[str, Any] | None) -> set[str]:
    if not isinstance(item, dict):
        return set()
    return {sig for raw in item.get("Applied_Law_Clauses") or [] if (sig := normalize_signature(raw))}


def article_set(signatures: set[str]) -> set[str]:
    return {sig.split("-")[0] for sig in signatures if sig}


def khoan_set(signatures: set[str]) -> set[str]:
    out: set[str] = set()
    for sig in signatures:
        parts = [part for part in sig.split("-") if part]
        if not parts:
            continue
        out.add("-".join(parts[:2]) if len(parts) >= 2 else parts[0])
    return out


def set_prf(pred: set[str], gt: set[str]) -> dict[str, float | int]:
    tp = len(pred & gt)
    fp = len(pred - gt)
    fn = len(gt - pred)
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {"tp": tp, "fp": fp, "fn": fn, "precision": precision, "recall": recall, "f1": f1}


def safe_mean(values: list[float | int | None]) -> float | None:
    clean = [float(v) for v in values if v is not None and not pd.isna(v)]
    return mean(clean) if clean else None


def safe_median(values: list[float | int | None]) -> float | None:
    clean = [float(v) for v in values if v is not None and not pd.isna(v)]
    return median(clean) if clean else None


def rate(numerator: float, denominator: float) -> float | None:
    return numerator / denominator if denominator else None


def months_from(item: dict[str, Any] | None, metric_item: dict[str, Any] | None, side: str) -> int:
    if isinstance(metric_item, dict):
        sentence_metrics = metric_item.get("phat_tu_months") or {}
        if side in sentence_metrics:
            return int(sentence_metrics.get(side) or 0)
    if not isinstance(item, dict):
        return 0
    return int(extract_imprisonment_months(item.get("Phat_Tu")))


def load_test_index(test_dir: Path) -> dict[str, dict[str, Any]]:
    index: dict[str, dict[str, Any]] = {}
    for path in sorted(test_dir.rglob("*.json")):
        try:
            data = load_json(path)
        except Exception:
            continue
        index[path.name] = {"path": path, "data": data}
    return index


def flatten_usage(doc: dict[str, Any]) -> dict[str, Any]:
    usage = doc.get("_usage") or {}
    calls = usage.get("calls") if isinstance(usage, dict) else None
    if isinstance(calls, list):
        return {
            "llm_calls": len(calls),
            "input_tokens": sum(int(call.get("prompt_tokens") or 0) for call in calls),
            "output_tokens": sum(int(call.get("completion_tokens") or 0) for call in calls),
            "total_tokens": sum(int(call.get("total_tokens") or 0) for call in calls),
            "used_providers": sorted({str(call.get("provider")) for call in calls if call.get("provider")}),
            "used_models": sorted({str(call.get("model")) for call in calls if call.get("model")}),
            "fallback_errors": sum(len(call.get("fallback_errors") or []) for call in calls),
        }
    if isinstance(usage, dict) and any(key in usage for key in ("prompt_tokens", "completion_tokens", "total_tokens")):
        return {
            "llm_calls": 1,
            "input_tokens": int(usage.get("prompt_tokens") or 0),
            "output_tokens": int(usage.get("completion_tokens") or 0),
            "total_tokens": int(usage.get("total_tokens") or 0),
            "used_providers": [str(usage.get("provider"))] if usage.get("provider") else [],
            "used_models": [str(usage.get("model"))] if usage.get("model") else [],
            "fallback_errors": len(usage.get("fallback_errors") or []),
        }
    return {"llm_calls": 0, "input_tokens": 0, "output_tokens": 0, "total_tokens": 0, "used_providers": [], "used_models": [], "fallback_errors": 0}


def fallback_used(doc: dict[str, Any], report: dict[str, Any]) -> bool:
    config = report.get("config") or {}
    requested_provider = str(config.get("provider") or "")
    requested_model = str(config.get("model") or "")
    llm = doc.get("llm") or {}
    if llm:
        return bool(
            str(llm.get("used_provider") or "") != str(llm.get("requested_provider") or "")
            or str(llm.get("used_model") or "") != str(llm.get("requested_model") or "")
        )
    usage_info = flatten_usage(doc)
    used_providers = set(usage_info["used_providers"])
    used_models = set(usage_info["used_models"])
    provider_changed = bool(used_providers and requested_provider and used_providers != {requested_provider})
    model_changed = bool(used_models and requested_model and used_models != {requested_model})
    return provider_changed or model_changed

## Load Reports And Build Rows

In [ ]:
reports: dict[str, dict[str, Any]] = {}
missing_paths = []
for model_name, path in RESULT_FILES.items():
    if path.exists():
        reports[model_name] = load_json(path)
    else:
        missing_paths.append(path)

if missing_paths:
    print("Missing result files:")
    for path in missing_paths:
        print(f"- {path}")

test_index = load_test_index(TEST_DIR)
print(f"Loaded {len(reports)} reports and {len(test_index)} ground-truth test files.")
display(pd.DataFrame([
    {
        "model": model,
        "path": str(RESULT_FILES[model]),
        "n_per_doc": len(report.get("per_doc") or []),
        "test_dir": (report.get("config") or {}).get("test_dir"),
        "input_fields": ",".join((report.get("config") or {}).get("input_fields") or []),
        "query_fields": ",".join((report.get("config") or {}).get("query_fields") or []),
    }
    for model, report in reports.items()
]))

In [ ]:
doc_rows: list[dict[str, Any]] = []
def_rows: list[dict[str, Any]] = []

for model_name, report in reports.items():
    for doc in report.get("per_doc") or []:
        usage_info = flatten_usage(doc)
        source_file = doc.get("source_file") or ""
        alignment = doc.get("defendant_alignment") or {}
        defendants = doc.get("defendants") or []
        status = doc.get("status") or ""
        reason = doc.get("reason") or ""
        gt_count = int(alignment.get("matched_count") or 0) + int(alignment.get("gt_only_count") or 0)
        pred_count = int(alignment.get("matched_count") or 0) + int(alignment.get("pred_only_count") or 0)
        if not alignment:
            gt_count = sum(1 for item in defendants if item.get("ground_truth"))
            pred_count = sum(1 for item in defendants if item.get("prediction"))
        doc_rows.append({
            "model": model_name,
            "doc_id": doc.get("doc_id"),
            "source_file": source_file,
            "status": status,
            "reason": reason,
            "gt_file_exists": source_file in test_index,
            "matched_count": int(alignment.get("matched_count") or 0),
            "gt_only_count": int(alignment.get("gt_only_count") or 0),
            "pred_only_count": int(alignment.get("pred_only_count") or 0),
            "gt_defendant_count": gt_count,
            "pred_defendant_count": pred_count,
            "llm_calls": usage_info["llm_calls"],
            "input_tokens": usage_info["input_tokens"],
            "output_tokens": usage_info["output_tokens"],
            "total_tokens": usage_info["total_tokens"],
            "fallback_errors": usage_info["fallback_errors"],
            "fallback_used": fallback_used(doc, report),
        })

        for idx, defendant in enumerate(defendants):
            gt = defendant.get("ground_truth") or {}
            pred = defendant.get("prediction") or {}
            metrics = defendant.get("metrics") or {}
            gt_full = signature_set(gt)
            pred_full = signature_set(pred)
            gt_article = article_set(gt_full)
            pred_article = article_set(pred_full)
            gt_khoan = khoan_set(gt_full)
            pred_khoan = khoan_set(pred_full)
            article_prf = set_prf(pred_article, gt_article)
            khoan_prf = set_prf(pred_khoan, gt_khoan)
            full_prf = set_prf(pred_full, gt_full)
            gt_mandatory = gt_article & MANDATORY_SUPPORT_DIEU
            pred_mandatory = pred_article & MANDATORY_SUPPORT_DIEU
            mandatory_recall = None if not gt_mandatory else len(gt_mandatory & pred_mandatory) / len(gt_mandatory)
            gt_offense = gt_article - SUPPORT_DIEU
            pred_offense = pred_article - SUPPORT_DIEU
            offense_prf = set_prf(pred_offense, gt_offense)
            gt_months = months_from(gt, metrics, "ground_truth")
            pred_months = months_from(pred, metrics, "prediction")
            sentence_error = pred_months - gt_months
            def_rows.append({
                "model": model_name,
                "doc_id": doc.get("doc_id"),
                "source_file": source_file,
                "defendant_index": idx,
                "status": status,
                "Bi_Cao": defendant.get("Bi_Cao") or (gt.get("Bi_Cao") or pred.get("Bi_Cao")),
                "has_ground_truth": bool(gt),
                "has_prediction": bool(pred),
                "gt_full": sorted(gt_full),
                "pred_full": sorted(pred_full),
                "gt_article": sorted(gt_article),
                "pred_article": sorted(pred_article),
                "article_precision": article_prf["precision"],
                "article_recall": article_prf["recall"],
                "article_f1": article_prf["f1"],
                "article_exact_match": gt_article == pred_article,
                "article_tp": article_prf["tp"],
                "article_fp": article_prf["fp"],
                "article_fn": article_prf["fn"],
                "khoan_precision": khoan_prf["precision"],
                "khoan_recall": khoan_prf["recall"],
                "khoan_f1": khoan_prf["f1"],
                "full_signature_precision": full_prf["precision"],
                "full_signature_recall": full_prf["recall"],
                "full_signature_f1": full_prf["f1"],
                "mandatory_support_article_recall": mandatory_recall,
                "offense_article_precision": offense_prf["precision"],
                "offense_article_recall": offense_prf["recall"],
                "offense_article_f1": offense_prf["f1"],
                "gt_months": gt_months,
                "pred_months": pred_months,
                "sentence_error_months": sentence_error,
                "sentence_abs_error_months": abs(sentence_error),
                "sentence_exact_match": gt_months == pred_months,
                "sentence_within_3_months": abs(sentence_error) <= 3,
                "sentence_within_6_months": abs(sentence_error) <= 6,
                "sentence_within_12_months": abs(sentence_error) <= 12,
                "custodial_class_match": (gt_months > 0) == (pred_months > 0),
                "zero_month_error": (gt_months == 0) != (pred_months == 0),
                "gt_Toi_Danh": gt.get("Toi_Danh", ""),
                "pred_Toi_Danh": pred.get("Toi_Danh", ""),
                "toi_danh_present": nonempty(pred.get("Toi_Danh")),
                "gt_toi_danh_normalized": normalize_toi_danh(gt.get("Toi_Danh")),
                "pred_toi_danh_normalized": normalize_toi_danh(pred.get("Toi_Danh")),
                "toi_danh_exact_match": normalize_toi_danh(gt.get("Toi_Danh")) == normalize_toi_danh(pred.get("Toi_Danh")),
                "toi_danh_partial_match": toi_danh_partial_match(pred.get("Toi_Danh"), gt.get("Toi_Danh")),
                "toi_danh_exact_normalized_match": normalize_text(gt.get("Toi_Danh")) == normalize_text(pred.get("Toi_Danh")),
                "toi_danh_token_f1": token_f1(pred.get("Toi_Danh"), gt.get("Toi_Danh")),
                "phat_tu_present": nonempty(pred.get("Phat_Tu")),
                "phat_tien_present": nonempty(pred.get("Phat_Tien")),
                "gt_phat_tien_present": nonempty(gt.get("Phat_Tien")),
                "phat_tien_exact_normalized_match": normalize_text(gt.get("Phat_Tien")) == normalize_text(pred.get("Phat_Tien")),
                "trach_nhiem_dan_su_present": nonempty(pred.get("Trach_Nhiem_Dan_Su")),
                "gt_trach_nhiem_dan_su_present": nonempty(gt.get("Trach_Nhiem_Dan_Su")),
                "trach_nhiem_dan_su_token_f1": token_f1(pred.get("Trach_Nhiem_Dan_Su"), gt.get("Trach_Nhiem_Dan_Su")),
                "xu_ly_vat_chung_present": nonempty(pred.get("Xu_Ly_Vat_Chung")),
                "gt_xu_ly_vat_chung_present": nonempty(gt.get("Xu_Ly_Vat_Chung")),
                "xu_ly_vat_chung_token_f1": token_f1(pred.get("Xu_Ly_Vat_Chung"), gt.get("Xu_Ly_Vat_Chung")),
                "applied_law_nonempty": bool(pred_full),
            })

doc_df = pd.DataFrame(doc_rows)
def_df = pd.DataFrame(def_rows)

display(doc_df.head())
display(def_df.head())

## Aggregate Metrics

In [ ]:
def aggregate_metrics(model_name: str, report: dict[str, Any]) -> dict[str, Any]:
    summary = report.get("summary") or {}
    summary_metrics = summary.get("metrics") or {}
    docs = doc_df[doc_df["model"] == model_name].copy()
    defs = def_df[(def_df["model"] == model_name) & (def_df["status"] == "processed")].copy()
    n_total = int(summary.get("n_total") or len(docs))
    n_processed = int(summary.get("n_processed") or (docs["status"] == "processed").sum())
    n_failed = int(summary.get("n_failed") or (docs["status"] == "failed").sum())
    n_skipped = int(summary.get("n_skipped") or (docs["status"] == "skipped").sum())
    gt_total = docs["gt_defendant_count"].sum()
    pred_total = docs["pred_defendant_count"].sum()
    article_pred_total = defs["article_tp"].sum() + defs["article_fp"].sum()
    article_gt_total = defs["article_tp"].sum() + defs["article_fn"].sum()
    return {
        "model": model_name,
        "n_total": n_total,
        "n_processed": n_processed,
        "n_failed": n_failed,
        "n_skipped": n_skipped,
        "processed_rate": rate(n_processed, n_total),
        "failure_rate": rate(n_failed, n_total),
        "skip_rate": rate(n_skipped, n_total),
        "resume_completeness": rate(docs["source_file"].nunique(), len(test_index)),
        "existing_law_article_precision_macro": summary_metrics.get("law_clause_set_precision_macro"),
        "existing_law_article_recall_macro": summary_metrics.get("law_clause_set_recall_macro"),
        "existing_law_article_f1_macro": summary_metrics.get("law_clause_set_f1_macro"),
        "existing_sentence_rmse_months_macro": summary_metrics.get("phat_tu_rmse_months_macro"),
        "per_defendant_article_f1_macro": safe_mean(defs["article_f1"].tolist()),
        "exact_article_set_match_rate": safe_mean(defs["article_exact_match"].astype(float).tolist()),
        "sentence_exact_match_rate": safe_mean(defs["sentence_exact_match"].astype(float).tolist()),
        "sentence_mae_months": safe_mean(defs["sentence_abs_error_months"].tolist()),
        "sentence_median_ae_months": safe_median(defs["sentence_abs_error_months"].tolist()),
        "sentence_within_3_month_rate": safe_mean(defs["sentence_within_3_months"].astype(float).tolist()),
        "sentence_within_6_month_rate": safe_mean(defs["sentence_within_6_months"].astype(float).tolist()),
        "sentence_within_12_month_rate": safe_mean(defs["sentence_within_12_months"].astype(float).tolist()),
        "sentence_bias_months": safe_mean(defs["sentence_error_months"].tolist()),
        "custodial_class_accuracy": safe_mean(defs["custodial_class_match"].astype(float).tolist()),
        "zero_month_error_rate": safe_mean(defs["zero_month_error"].astype(float).tolist()),
        "defendant_name_match_rate": rate(docs["matched_count"].sum(), gt_total),
        "missing_defendant_rate": rate(docs["gt_only_count"].sum(), gt_total),
        "extra_defendant_rate": rate(docs["pred_only_count"].sum(), pred_total),
        "full_signature_precision_macro": safe_mean(defs["full_signature_precision"].tolist()),
        "full_signature_recall_macro": safe_mean(defs["full_signature_recall"].tolist()),
        "full_signature_f1_macro": safe_mean(defs["full_signature_f1"].tolist()),
        "khoan_level_precision_macro": safe_mean(defs["khoan_precision"].tolist()),
        "khoan_level_recall_macro": safe_mean(defs["khoan_recall"].tolist()),
        "khoan_level_f1_macro": safe_mean(defs["khoan_f1"].tolist()),
        "mandatory_support_article_recall": safe_mean(defs["mandatory_support_article_recall"].tolist()),
        "offense_article_precision_macro": safe_mean(defs["offense_article_precision"].tolist()),
        "offense_article_recall_macro": safe_mean(defs["offense_article_recall"].tolist()),
        "offense_article_f1_macro": safe_mean(defs["offense_article_f1"].tolist()),
        "overcitation_rate": rate(defs["article_fp"].sum(), article_pred_total),
        "undercitation_rate": rate(defs["article_fn"].sum(), article_gt_total),
        "toi_danh_present_rate": safe_mean(defs["toi_danh_present"].astype(float).tolist()),
        "toi_danh_exact_match_rate": safe_mean(defs["toi_danh_exact_match"].astype(float).tolist()),
        "toi_danh_partial_match_rate": safe_mean(defs["toi_danh_partial_match"].astype(float).tolist()),
        "phat_tu_present_rate": safe_mean(defs["phat_tu_present"].astype(float).tolist()),
        "phat_tien_present_when_gt_rate": safe_mean(defs.loc[defs["gt_phat_tien_present"], "phat_tien_present"].astype(float).tolist()),
        "trach_nhiem_dan_su_present_when_gt_rate": safe_mean(defs.loc[defs["gt_trach_nhiem_dan_su_present"], "trach_nhiem_dan_su_present"].astype(float).tolist()),
        "xu_ly_vat_chung_present_when_gt_rate": safe_mean(defs.loc[defs["gt_xu_ly_vat_chung_present"], "xu_ly_vat_chung_present"].astype(float).tolist()),
        "applied_law_nonempty_rate": safe_mean(defs["applied_law_nonempty"].astype(float).tolist()),
        "toi_danh_exact_normalized_match_rate": safe_mean(defs["toi_danh_exact_normalized_match"].astype(float).tolist()),
        "toi_danh_token_f1": safe_mean(defs["toi_danh_token_f1"].tolist()),
        "phat_tien_exact_normalized_match_rate": safe_mean(defs["phat_tien_exact_normalized_match"].astype(float).tolist()),
        "trach_nhiem_dan_su_token_f1": safe_mean(defs["trach_nhiem_dan_su_token_f1"].tolist()),
        "xu_ly_vat_chung_token_f1": safe_mean(defs["xu_ly_vat_chung_token_f1"].tolist()),
        "avg_input_tokens": safe_mean(docs["input_tokens"].tolist()),
        "avg_output_tokens": safe_mean(docs["output_tokens"].tolist()),
        "avg_total_tokens": safe_mean(docs["total_tokens"].tolist()),
        "total_input_tokens": docs["input_tokens"].sum(),
        "total_output_tokens": docs["output_tokens"].sum(),
        "total_tokens": docs["total_tokens"].sum(),
        "avg_llm_calls_per_doc": safe_mean(docs["llm_calls"].tolist()),
        "fallback_rate": safe_mean(docs["fallback_used"].astype(float).tolist()),
        "parse_failure_rate": safe_mean((docs["reason"] == "parse_error").astype(float).tolist()),
        "generation_failure_rate": safe_mean((docs["reason"] == "generation_error").astype(float).tolist()),
    }


metrics_df = pd.DataFrame([aggregate_metrics(model, report) for model, report in reports.items()]).set_index("model")
display(metrics_df.round(4))

EXPORT_DIR.mkdir(parents=True, exist_ok=True)
metrics_df.to_csv(EXPORT_DIR / "summary_metrics.csv")
doc_df.to_csv(EXPORT_DIR / "doc_rows.csv", index=False)
def_df.to_csv(EXPORT_DIR / "defendant_rows.csv", index=False)
print(f"Exported CSVs to {EXPORT_DIR}")

## Quick Metric Views

In [ ]:
headline_cols = [
    "processed_rate",
    "existing_law_article_f1_macro",
    "full_signature_f1_macro",
    "offense_article_f1_macro",
    "mandatory_support_article_recall",
    "exact_article_set_match_rate",
    "toi_danh_exact_match_rate",
    "toi_danh_partial_match_rate",
    "sentence_mae_months",
    "existing_sentence_rmse_months_macro",
    "sentence_within_6_month_rate",
    "defendant_name_match_rate",
    "missing_defendant_rate",
    "avg_input_tokens",
    "avg_output_tokens",
    "avg_total_tokens",
    "fallback_rate",
]

headline_df = metrics_df[headline_cols].copy()
display(headline_df.round(4).T)

## Basic Plots

In [ ]:
law_plot_cols = [
    "existing_law_article_f1_macro",
    "khoan_level_f1_macro",
    "full_signature_f1_macro",
    "offense_article_f1_macro",
]

ax = metrics_df[law_plot_cols].plot(kind="bar", figsize=(10, 5), rot=0)
ax.set_title("Law Metric Comparison")
ax.set_ylabel("F1")
ax.set_ylim(0, 1)
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

In [ ]:
toi_danh_plot_cols = ["toi_danh_exact_match_rate", "toi_danh_partial_match_rate"]
ax = metrics_df[toi_danh_plot_cols].plot(kind="bar", figsize=(9, 5), rot=0)
ax.set_title("Toi_Danh Match Rates")
ax.set_ylabel("Rate")
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

In [ ]:
sentence_plot_cols = [
    "sentence_mae_months",
    "sentence_median_ae_months",
    "existing_sentence_rmse_months_macro",
]

ax = metrics_df[sentence_plot_cols].plot(kind="bar", figsize=(10, 5), rot=0)
ax.set_title("Sentence Error Comparison")
ax.set_ylabel("Months")
plt.tight_layout()
plt.show()

ax = metrics_df[["sentence_within_3_month_rate", "sentence_within_6_month_rate", "sentence_within_12_month_rate"]].plot(
    kind="bar", figsize=(10, 5), rot=0
)
ax.set_title("Sentence Accuracy Within Tolerance")
ax.set_ylabel("Rate")
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

In [ ]:
plot_df = def_df[def_df["status"] == "processed"].copy()
models = list(reports.keys())
box_data = [plot_df.loc[plot_df["model"] == model, "sentence_abs_error_months"].dropna() for model in models]

fig, ax = plt.subplots(figsize=(9, 5))
ax.boxplot(box_data, labels=models, showfliers=False)
ax.set_title("Per-Defendant Sentence Absolute Error")
ax.set_ylabel("Absolute Error (months)")
plt.tight_layout()
plt.show()

In [ ]:
alignment_cols = ["defendant_name_match_rate", "missing_defendant_rate", "extra_defendant_rate"]
ax = metrics_df[alignment_cols].plot(kind="bar", figsize=(10, 5), rot=0)
ax.set_title("Defendant Alignment")
ax.set_ylabel("Rate")
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

token_cols = ["avg_input_tokens", "avg_output_tokens"]
ax = metrics_df[token_cols].plot(kind="bar", figsize=(10, 5), rot=0)
ax.set_title("Average Input vs Output Tokens Per Document")
ax.set_ylabel("Tokens")
plt.tight_layout()
plt.show()

## Drill-Down Tables

In [ ]:
# Worst sentence errors by model.
display(
    def_df.sort_values("sentence_abs_error_months", ascending=False)[
        [
            "model",
            "source_file",
            "Bi_Cao",
            "gt_months",
            "pred_months",
            "sentence_abs_error_months",
            "gt_full",
            "pred_full",
        ]
    ].head(30)
)

# Lowest strict legal signature F1 by model.
display(
    def_df.sort_values("full_signature_f1", ascending=True)[
        ["model", "source_file", "Bi_Cao", "full_signature_f1", "gt_full", "pred_full"]
    ].head(30)
)